# Coulomb Archive — 시각적 데모

가설 ELG 는 잠시 잊고, **2D 좌표 위의 샘플 데이터**로 Coulomb Archive 를 돌린다. 목적은 두 가지.

1. **거리 개념을 눈으로**: 유클리드 거리 = 두 데이터 포인트 사이 직관적 거리
2. **아카이브의 셀 형성 과정**: 스트리밍으로 데이터가 들어올 때 아카이브가 공간을 어떻게 분할하는지 시각화

핵심 관찰 포인트:
- 시간 흐름에 따른 아카이브 진화 (time-lapse)
- 최종 아카이브의 **Voronoi 셀** (각 멤버가 지배하는 영역)
- **Potential field** 등고선 (반발장의 지형)
- γ 다이얼이 다양성-품질 밸런스에 미치는 영향

의존성: numpy + matplotlib 만.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap

# macOS 한글 폰트 (Linux는 NanumGothic 등으로 교체)
plt.rcParams['font.family'] = ['AppleGothic', 'NanumGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(7)


## 1. 샘플 데이터 생성

2D 평면 `[0, 10] × [0, 10]` 위에 **500 개의 후보 데이터 포인트**를 뿌린다. 각 포인트는:

- **위치** `(x, y)` — 유클리드 거리 계산의 기반
- **quality score** `q ∈ [0, 1]` — 4개의 '언덕(peak)'을 갖는 gaussian mixture landscape 위에서 뽑음

즉 특정 영역은 고품질 밀집, 나머지는 저품질 산발. 실제 가설 진화에서 LLM 이 특정 도메인 근처를 반복 탐색하는 상황과 유사.

In [ ]:
# Quality landscape: 4 개의 gaussian peak
PEAKS = [
    (2.0, 2.0, 0.95, 0.9),   # (cx, cy, height, width)
    (7.5, 2.5, 0.85, 0.7),
    (3.0, 7.5, 0.90, 1.0),
    (8.0, 8.0, 0.80, 0.8),
]

def quality_at(x, y):
    q = 0.0
    for cx, cy, h, w in PEAKS:
        q += h * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * w ** 2))
    return q

# 500 포인트 uniform sampling + quality landscape evaluation + noise
N_POINTS = 500
positions = np.random.uniform(0.5, 9.5, size=(N_POINTS, 2))
qualities = np.array([quality_at(p[0], p[1]) for p in positions])
qualities += np.random.normal(0, 0.03, size=N_POINTS)
qualities = np.clip(qualities, 0.05, 1.0)

# Landscape 자체를 시각화
gx, gy = np.meshgrid(np.linspace(0, 10, 300), np.linspace(0, 10, 300))
Q_field = np.zeros_like(gx)
for cx, cy, h, w in PEAKS:
    Q_field += h * np.exp(-((gx - cx) ** 2 + (gy - cy) ** 2) / (2 * w ** 2))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cs = axes[0].contourf(gx, gy, Q_field, levels=20, cmap='YlOrRd')
axes[0].set_title('Quality landscape (참: 4 peaks)')
axes[0].set_xlim(0, 10); axes[0].set_ylim(0, 10)
plt.colorbar(cs, ax=axes[0], label='quality')

sc = axes[1].scatter(positions[:, 0], positions[:, 1], c=qualities, s=15,
                     cmap='YlOrRd', edgecolors='gray', linewidths=0.3)
axes[1].set_title(f'{N_POINTS} 개 후보 데이터 (색 = quality)')
axes[1].set_xlim(0, 10); axes[1].set_ylim(0, 10)
plt.colorbar(sc, ax=axes[1], label='quality')
plt.tight_layout(); plt.show()

## 2. 거리 정의

가장 직관적인 거리: **유클리드 거리**.

```
d(p_i, p_j) = √((x_i - x_j)² + (y_i - y_j)²)
```

실전 ELG 에서는 이 자리에 tree kernel 기반 `1 - K_tree` 가 들어가지만, 시각화 목적으로는 유클리드가 훨씬 명료함.

In [ ]:
def distance(p1, p2):
    return float(np.linalg.norm(p1 - p2))

## 3. Coulomb Archive 구현 (Streaming)

두 규칙만 있으면 끝.

**Rule 1** (부모 샘플링): `P(parent = h) ∝ score(h) · exp(-γ · U(h; A))`

**Rule 2** (아카이빙): 새 후보 h*의 net contribution `Δ(h*) = score(h*) - γ · U(h*; A)`
- 아카이브가 비어있으면 편입
- 꽉 찼으면 **가장 낮은 Δ 를 가진 기존 멤버와 비교** → 후보가 나으면 축출·편입

매 iteration 마다 스냅샷을 남겨서 나중에 time-lapse 시각화.

In [ ]:
class CoulombArchive:
    def __init__(self, capacity, gamma, eps=0.05):
        self.cap = capacity
        self.gamma = gamma
        self.eps = eps
        self.members = []           # list of (orig_idx, pos, quality)
        self.snapshots = []         # 각 step 후 members 복사본
        self.events = []            # ('admit'/'evict_admit'/'reject', ...)

    def _potential_at(self, pos, exclude_slot=None):
        u = 0.0
        for i, (_, mp, mq) in enumerate(self.members):
            if i == exclude_slot:
                continue
            d2 = float(np.sum((pos - mp) ** 2))
            u += mq / (d2 + self.eps)
        return u

    def _delta_of_slot(self, slot):
        _, mp, mq = self.members[slot]
        return mq - self.gamma * self._potential_at(mp, exclude_slot=slot)

    def offer(self, orig_idx, pos, quality):
        if len(self.members) < self.cap:
            self.members.append((orig_idx, pos, quality))
            self.events.append(('admit', orig_idx, None))
        else:
            # 가장 약한 기존 멤버 찾기
            deltas = [self._delta_of_slot(s) for s in range(len(self.members))]
            weak_slot = int(np.argmin(deltas))
            weak_delta = deltas[weak_slot]
            weak_idx = self.members[weak_slot][0]
            # 후보가 weak_slot 을 대체했다고 가정했을 때의 Δ
            cand_delta = quality - self.gamma * self._potential_at(pos, exclude_slot=weak_slot)
            if cand_delta > weak_delta:
                self.members[weak_slot] = (orig_idx, pos, quality)
                self.events.append(('evict_admit', orig_idx, weak_idx))
            else:
                self.events.append(('reject', orig_idx, None))
        self.snapshots.append([m for m in self.members])

## 4. 스트리밍 실행

500 개 데이터를 랜덤 순서로 하나씩 아카이브에 offer. 아카이브 상한은 15.

In [ ]:
CAPACITY = 15
GAMMA = 0.4

order = np.random.permutation(N_POINTS)
archive = CoulombArchive(capacity=CAPACITY, gamma=GAMMA)
for idx in order:
    archive.offer(int(idx), positions[idx], float(qualities[idx]))

# 이벤트 요약
from collections import Counter
counts = Counter(e[0] for e in archive.events)
print(f'총 offer: {len(archive.events)}')
for k, v in counts.items():
    print(f'  {k:<15} {v}')
print(f'\n최종 archive 크기: {len(archive.members)}')
print(f'평균 quality: {np.mean([m[2] for m in archive.members]):.3f}')

## 5. Time-lapse — 아카이브 진화

여러 시점에서 아카이브 상태를 스냅샷. 파란 X = 현재 아카이브 멤버, 회색 점 = 지금까지 본 후보들.

관찰 포인트: 초기에는 quality 높은 곳에 몰리다가, 시간이 지날수록 반발 규칙이 작동해 **공간 전체에 골고루 퍼진 4개 peak 대표들**이 형성됨.

In [ ]:
snapshot_steps = [15, 30, 60, 120, 250, 500]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, step in zip(axes.flat, snapshot_steps):
    idx_seen = order[:step]
    # 배경: quality landscape (연하게)
    ax.contourf(gx, gy, Q_field, levels=15, cmap='YlOrRd', alpha=0.35)
    # 지금까지 본 후보들 (연회색)
    ax.scatter(positions[idx_seen, 0], positions[idx_seen, 1],
               c='lightgray', s=8, alpha=0.5, zorder=2)
    # 현재 archive 멤버
    members = archive.snapshots[step - 1]
    if members:
        mpts = np.array([m[1] for m in members])
        mqs  = np.array([m[2] for m in members])
        ax.scatter(mpts[:, 0], mpts[:, 1], c=mqs, cmap='viridis',
                   s=250, edgecolors='navy', linewidths=2, marker='X',
                   vmin=0, vmax=1, zorder=5)
    ax.set_title(f'Step {step}  |A|={len(members)}  γ={GAMMA}', fontsize=11)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_aspect('equal')
plt.suptitle('Coulomb Archive time-lapse — 배경 heatmap = quality landscape',
             fontsize=13, y=1.00)
plt.tight_layout(); plt.show()

## 6. 아카이브 셀 형성 — Voronoi 분할

최종 아카이브 멤버 각각이 공간에서 지배하는 영역을 **가장 가까운 멤버로 색칠**해서 표현. 이게 곧 Coulomb Archive 가 만들어내는 **암묵적 셀 구조**.

MAP-Elites 는 격자를 사람이 미리 그어두지만, Coulomb 은 아카이브 멤버들이 스스로 위치를 잡으면서 셀 경계가 emergent 하게 형성됨.

In [ ]:
final = archive.snapshots[-1]
final_pts = np.array([m[1] for m in final])
final_qs  = np.array([m[2] for m in final])

# 그리드 위 각 픽셀을 가장 가까운 archive 멤버로 매핑
grid_pts = np.stack([gx.ravel(), gy.ravel()], axis=1)
dists_to_members = np.linalg.norm(
    grid_pts[:, None, :] - final_pts[None, :, :], axis=2
)  # (n_grid, n_members)
nearest = np.argmin(dists_to_members, axis=1).reshape(gx.shape)

# 각 셀에 pastel 색
n_members = len(final_pts)
cmap = plt.get_cmap('tab20', n_members)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(nearest, extent=(0, 10, 0, 10), origin='lower',
          cmap=cmap, alpha=0.6, aspect='equal')
# 후보 점들 (지금까지 본 것 = 전부)
ax.scatter(positions[:, 0], positions[:, 1], c=qualities, cmap='Greys',
           s=8, alpha=0.4, zorder=2)
# 아카이브 멤버
ax.scatter(final_pts[:, 0], final_pts[:, 1], c='white', edgecolors='black',
           s=350, linewidths=2.5, marker='X', zorder=5)
for i, (_, pos, q) in enumerate(final):
    ax.annotate(f'{i}\nq={q:.2f}', (pos[0], pos[1]), fontsize=8,
                ha='center', va='center', xytext=(0, -25),
                textcoords='offset points', color='black',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
ax.set_title(f'최종 아카이브의 셀 분할 (Voronoi) — {n_members} 멤버', fontsize=13)
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
plt.tight_layout(); plt.show()

## 7. Coulomb Potential Field

아카이브 멤버들이 만드는 **반발장**의 지형을 등고선으로. 어두운 골짜기 = 새 후보가 편입되기 유리한 지대, 밝은 봉우리 = 이미 붐빈 exclusion zone.

In [ ]:
# grid 위 potential 계산
U_field = np.zeros_like(gx)
for _, mp, mq in final:
    d2 = (gx - mp[0]) ** 2 + (gy - mp[1]) ** 2
    U_field += mq / (d2 + 0.05)
U_log = np.log1p(U_field)

fig, ax = plt.subplots(figsize=(11, 9))
cs = ax.contourf(gx, gy, U_log, levels=30, cmap='magma')
ax.contour(gx, gy, U_log, levels=12, colors='white', linewidths=0.5, alpha=0.5)
ax.scatter(positions[:, 0], positions[:, 1], c='cyan', s=6, alpha=0.3, zorder=2)
ax.scatter(final_pts[:, 0], final_pts[:, 1], c='cyan', edgecolors='black',
           s=280, linewidths=2, marker='X', zorder=5, label='archive')
ax.set_title(f'Coulomb potential field  U(x)  —  log(1+U) scale', fontsize=13)
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.legend(loc='upper right')
plt.colorbar(cs, ax=ax, label='log(1 + U)')
plt.tight_layout(); plt.show()

## 8. γ 다이얼의 효과

γ 하나만 바꿔서 스트리밍을 재실행. γ 가 클수록 **다양성 위주**, 작을수록 **점수 위주**.

- `γ = 0`: 반발 없음. 그냥 top-K by score → 4개 peak 중 한두 개에 몰림
- `γ = 0.1`: 약한 반발
- `γ = 0.4`: 균형 (기본값)
- `γ = 2.0`: 강한 반발 → 공간 전체에 골고루 퍼짐 (품질 다소 희생)

In [ ]:
gammas_sweep = [0.0, 0.1, 0.4, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, g in zip(axes, gammas_sweep):
    arc = CoulombArchive(capacity=CAPACITY, gamma=g)
    for idx in order:
        arc.offer(int(idx), positions[idx], float(qualities[idx]))
    fmembers = arc.snapshots[-1]
    fpts = np.array([m[1] for m in fmembers])
    fqs  = np.array([m[2] for m in fmembers])
    ax.contourf(gx, gy, Q_field, levels=15, cmap='YlOrRd', alpha=0.4)
    ax.scatter(positions[:, 0], positions[:, 1], c='lightgray', s=5, alpha=0.4)
    ax.scatter(fpts[:, 0], fpts[:, 1], c=fqs, cmap='viridis',
               s=250, edgecolors='navy', linewidths=2, marker='X',
               vmin=0, vmax=1)
    # 다양성 지표: 평균 최근접 이웃 거리
    if len(fpts) > 1:
        pd = np.linalg.norm(fpts[:, None] - fpts[None], axis=2)
        np.fill_diagonal(pd, np.inf)
        nn_dist = float(np.mean(np.min(pd, axis=1)))
    else:
        nn_dist = 0.0
    mean_q = float(np.mean(fqs))
    ax.set_title(f'γ = {g}\nmean-NN dist = {nn_dist:.2f},  mean q = {mean_q:.2f}',
                 fontsize=11)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_aspect('equal')

plt.suptitle('γ 다이얼 스윕 — 반발 강도만 바꿔서 다양성-품질 균형 이동',
             fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

## 9. Score-only 베이스라인과 비교

만약 아카이브가 그냥 **top-K by score** 만 유지했다면? 반발 개념이 없어서 4 peak 중 한 곳에 대부분 몰릴 것.

In [ ]:
# Score-only: 그냥 quality 상위 K 개
top_k_idx = np.argsort(-qualities)[:CAPACITY]
top_k_pts = positions[top_k_idx]
top_k_qs  = qualities[top_k_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, (pts, qs, title) in zip(axes, [
    (top_k_pts, top_k_qs, 'Score-only baseline (top-K)'),
    (final_pts, final_qs, f'Coulomb Archive (γ={GAMMA})'),
]):
    ax.contourf(gx, gy, Q_field, levels=15, cmap='YlOrRd', alpha=0.4)
    ax.scatter(positions[:, 0], positions[:, 1], c='lightgray', s=5, alpha=0.4)
    ax.scatter(pts[:, 0], pts[:, 1], c=qs, cmap='viridis',
               s=250, edgecolors='navy', linewidths=2, marker='X',
               vmin=0, vmax=1)
    pd = np.linalg.norm(pts[:, None] - pts[None], axis=2)
    np.fill_diagonal(pd, np.inf)
    nn = float(np.mean(np.min(pd, axis=1)))
    mq = float(np.mean(qs))
    ax.set_title(f'{title}\nmean-NN dist = {nn:.2f},  mean q = {mq:.2f}', fontsize=12)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_aspect('equal')

plt.tight_layout(); plt.show()

print('Score-only: 특정 peak 근처에 뭉치는 경향 → mean-NN 거리 작음')
print('Coulomb   : 4 peak 대표를 골고루 커버 → mean-NN 거리 큼, quality 는 조금 희생')

## 10. 요약

이 노트북에서 시각적으로 확인한 것:

1. **스트리밍**: 데이터가 1개씩 들어와도 배치 재선택 없이 아카이브가 계속 유지·갱신됨.
2. **셀 형성 (emergent)**: 아카이브 멤버들이 스스로 위치를 잡으며 Voronoi 셀 구조를 만듦. MAP-Elites 처럼 격자를 미리 그을 필요 없음.
3. **Potential field**: 각 멤버 주변에 exclusion zone 이 형성되고, 그 사이 골짜기가 새 후보의 편입 지대가 됨.
4. **γ 다이얼 하나**로 exploration-exploitation 전 범위 조절.
5. **Score-only 대비**: Coulomb 은 다양성 상당히 개선, 평균 quality 는 조금만 희생.

이 데모의 유클리드 거리 자리에 tree kernel `1 - K_tree` 를 넣으면 그대로 HypoEvolve 의 가설 진화에 적용됨. 나머지는 동일.